# MadaKapPy: A tool for in situ biomass estimation of kappaphycus

cultures

Simon Oiry [](https://orcid.org/0000-0001-7161-5246) (Institut des Substances et Organismes de la Mer, ISOMer, Nantes Université, UR 2160, F-44000 Nantes, France, Consiglio Nazionale delle Ricerche, Istituto di Scienze Marine (CNR-ISMAR), 34149 Trieste, Italy)  
Sébastien Jan (Nosy Boraha Seaweed Company, Ilot Madame BP04-Ambodifotatra, Sainte-Marie 515, Madagascar)  
Manon Museux (Nosy Boraha Seaweed Company, Ilot Madame BP04-Ambodifotatra, Sainte-Marie 515, Madagascar)  
Ravo A. Mahandrisoa Randriamaroson (Nosy Boraha Seaweed Company, Ilot Madame BP04-Ambodifotatra, Sainte-Marie 515, Madagascar)  
Anthony (Nosy Boraha Seaweed Company, Ilot Madame BP04-Ambodifotatra, Sainte-Marie 515, Madagascar)  
Laurent Barillé [](https://orcid.org/0000-0001-5138-2684) (Institut des Substances et Organismes de la Mer, ISOMer, Nantes Université, UR 2160, F-44000 Nantes, France)  
April 24, 2026

To be Written

# 1. Introduction

# 2. Materiel & Methods

## 2.1 Study site

In [ ]:
library(terra)

terra 1.9.11

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     

── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ tidyr::extract() masks terra::extract()
✖ dplyr::filter()  masks stats::filter()
✖ dplyr::lag()     masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Sainte-Marie (also known as Nosy Boraha) is a small island located off the east coast of Madagascar in the Indian Ocean (<a href="#fig-study-area-map" class="quarto-xref">Figure 1</a>). The study area is situated near the village of Ankobahoba, on the eastern side of the island, within a narrow (15 km long × 1.5 km wide) and very shallow lagoon. Water depths generally range from \<1 to 5 m at low tide. The lagoon is enclosed by a barrier reef and connected to the open ocean through several reef passes. Its total water volume is approximately 30.3 million cubic meters. Substrates are dominated by sand and coral rubble, supporting seagrass meadows, macroalgal beds, and patch reefs, which become increasingly frequent toward the reef front.

Aquaculture represents a major and expanding use of the eastern lagoon, where seaweed farming has become a key component of coastal livelihoods. The activity is dominated by the cultivation of Kappaphycus alvarezii (“cottonii”), primarily in the northern part of the lagoon, using off-bottom longline techniques specifically adapted to very shallow environments. While this activity provides an important alternative source of income in a context of declining fish stocks, it also leads to increasing spatial competition with traditional fishing practices in this confined coastal system.

More broadly, the lagoon is now included within the newly established AMTP Sorkay (Marine and Terrestrial Protected Area), a large-scale conservation initiative (~265,000 ha) aimed at preserving the island’s marine biodiversity and promoting sustainable resource management through community-based governance.

In [ ]:
from pathlib import Path

import contextily as cx
import geopandas as gpd
import numpy as np
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import rasterio
from rasterio.mask import mask as raster_mask
from rasterio.transform import array_bounds
from rasterio.windows import Window, bounds as window_bounds, from_bounds
from matplotlib.gridspec import GridSpec
from scipy.ndimage import gaussian_filter
from shapely.geometry import box

PROJECT_ROOT = Path.cwd()
STUDY_AREA_CANDIDATES = [
    PROJECT_ROOT / "Data" / "mask_northernmost_lagoon.shp",
    PROJECT_ROOT / "Data" / "shp" / "mask_northernmost_lagoon_32739.shp",
]
STUDY_AREA = next((path for path in STUDY_AREA_CANDIDATES if path.exists()), None)
if STUDY_AREA is None:
    raise FileNotFoundError(
        "Study-area shapefile not found. Checked: "
        + ", ".join(str(path) for path in STUDY_AREA_CANDIDATES)
    )

CADASTER = PROJECT_ROOT / "Data" / "shp" / "Cadaster_NBS_Dec2025.shp"
if not CADASTER.exists():
    raise FileNotFoundError(f"Culture-plot cadaster not found: {CADASTER}")

ZOOM_STUDYSITE = PROJECT_ROOT / "Data" / "shp" / "zoom_studysite.shp"
if not ZOOM_STUDYSITE.exists():
    raise FileNotFoundError(f"Zoom study-site polygon not found: {ZOOM_STUDYSITE}")

DEPTH_CANDIDATES = [
    path
    for folder in (PROJECT_ROOT / "Data" / "depth", PROJECT_ROOT / "Data" / "Depth")
    for path in sorted(folder.glob("*.tif"))
]
DEPTH_RASTER = next(iter(DEPTH_CANDIDATES), None)
if DEPTH_RASTER is None:
    raise FileNotFoundError("No depth raster found in Data/depth or Data/Depth")

DRONE_DIR = PROJECT_ROOT / "Data" / "drone"
DRONE_CANDIDATES = [
    path
    for pattern in ("*.tif", "*.tiff")
    for path in sorted(DRONE_DIR.glob(pattern))
]
DRONE_RASTER = next(iter(DRONE_CANDIDATES), None)
if DRONE_RASTER is None:
    raise FileNotFoundError(f"No drone raster found in {DRONE_DIR}")

PICTURE_DIR = PROJECT_ROOT / "Data" / "Pictures"
PICTURE_NAMES = {
    "D": "DJI_20251205051926_0211_V.jpg",
    "E": "NADIR_plot.jpg",
    "F": "Kappa2.jpg",
}
PICTURE_FILES = {label: PICTURE_DIR / name for label, name in PICTURE_NAMES.items()}
missing_pictures = [str(path) for path in PICTURE_FILES.values() if not path.exists()]
if missing_pictures:
    raise FileNotFoundError(
        "Required picture(s) not found: " + ", ".join(missing_pictures)
    )

FIGURE_DIR = PROJECT_ROOT / "Figures"
FIGURE_DIR.mkdir(exist_ok=True)
OUT_FILE = FIGURE_DIR / "study_area.png"

REFERENCE_TILE_SOURCE = cx.providers.CartoDB.VoyagerNoLabels
SATELLITE_TILE_SOURCE = cx.providers.Esri.WorldImagery
DEPTH_SMOOTHING_SIGMA_PIXELS = 1.4


def bbox_wgs84_to_web_mercator(lon_min, lat_min, lon_max, lat_max):
    bbox = gpd.GeoDataFrame(
        geometry=[box(lon_min, lat_min, lon_max, lat_max)],
        crs="EPSG:4326",
    )
    return bbox.to_crs("EPSG:3857").total_bounds


def square_bounds(bounds, pad_fraction=0.12, min_size=None):
    xmin, ymin, xmax, ymax = bounds
    width = xmax - xmin
    height = ymax - ymin
    size = max(width, height) * (1 + 2 * pad_fraction)
    if min_size is not None:
        size = max(size, min_size)
    xmid = (xmin + xmax) / 2
    ymid = (ymin + ymax) / 2
    return [xmid - size / 2, ymid - size / 2, xmid + size / 2, ymid + size / 2]


def transform_bounds(bounds, src_crs, dst_crs):
    bbox = gpd.GeoDataFrame(geometry=[box(*bounds)], crs=src_crs)
    return bbox.to_crs(dst_crs).total_bounds


def add_basemap(ax, extent, zoom, source, crs="EPSG:3857"):
    ax.set_xlim(extent[0], extent[2])
    ax.set_ylim(extent[1], extent[3])
    cx.add_basemap(
        ax,
        source=source,
        crs=crs,
        zoom=zoom,
        attribution=False,
        reset_extent=False,
    )
    ax.set_axis_off()


def add_rgb_raster(ax, raster_path, extent=None, zorder=0):
    with rasterio.open(raster_path) as src:
        raster_extent = [src.bounds.left, src.bounds.bottom, src.bounds.right, src.bounds.top]
        target_extent = raster_extent if extent is None else [
            max(extent[0], raster_extent[0]),
            max(extent[1], raster_extent[1]),
            min(extent[2], raster_extent[2]),
            min(extent[3], raster_extent[3]),
        ]
        if target_extent[0] >= target_extent[2] or target_extent[1] >= target_extent[3]:
            raise ValueError(f"Requested extent is outside raster bounds: {raster_path}")

        window = from_bounds(*target_extent, transform=src.transform)
        window = window.round_offsets().round_lengths()
        window = window.intersection(Window(0, 0, src.width, src.height))
        rgb = src.read([1, 2, 3], window=window)
        image_extent = window_bounds(window, src.transform)

    ax.imshow(
        np.moveaxis(rgb, 0, -1),
        extent=[image_extent[0], image_extent[2], image_extent[1], image_extent[3]],
        origin="upper",
        interpolation="bilinear",
        zorder=zorder,
    )
    ax.set_xlim(target_extent[0], target_extent[2])
    ax.set_ylim(target_extent[1], target_extent[3])
    ax.set_axis_off()


def add_red_box(ax, bounds, linewidth=2.2):
    xmin, ymin, xmax, ymax = bounds
    ax.add_patch(
        mpatches.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            fill=False,
            edgecolor="#e31a1c",
            linewidth=linewidth,
            zorder=8,
        )
    )


def add_panel_label(ax, label, fontsize=13):
    ax.text(
        0.035,
        0.955,
        label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=fontsize,
        fontweight="bold",
        color="white",
        bbox={"boxstyle": "square,pad=0.25", "facecolor": "black", "alpha": 0.72, "edgecolor": "none"},
        zorder=10,
    )


def add_panel_border(ax, color="#222222", linewidth=0.7):
    ax.add_patch(
        mpatches.Rectangle(
            (0, 0),
            1,
            1,
            transform=ax.transAxes,
            fill=False,
            edgecolor=color,
            linewidth=linewidth,
            zorder=30,
        )
    )


def add_scale_bar(
    ax,
    length_m,
    label,
    x_fraction=0.58,
    y_fraction=0.06,
    fontsize=7,
):
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    xspan = xlim[1] - xlim[0]
    yspan = ylim[1] - ylim[0]
    x0 = xlim[0] + x_fraction * xspan
    y0 = ylim[0] + y_fraction * yspan
    x1 = x0 + length_m
    outline = [pe.Stroke(linewidth=4, foreground="black"), pe.Normal()]

    ax.plot(
        [x0, x1],
        [y0, y0],
        color="white",
        linewidth=2.4,
        solid_capstyle="butt",
        path_effects=outline,
        zorder=10,
    )
    ax.text(
        (x0 + x1) / 2,
        y0 + yspan * 0.025,
        label,
        ha="center",
        va="bottom",
        fontsize=fontsize,
        color="white",
        path_effects=[pe.Stroke(linewidth=2.0, foreground="black"), pe.Normal()],
        zorder=10,
    )


def center_crop_to_ratio(image, target_ratio):
    height, width = image.shape[:2]
    image_ratio = width / height

    if image_ratio > target_ratio:
        crop_width = int(round(height * target_ratio))
        x0 = (width - crop_width) // 2
        return image[:, x0 : x0 + crop_width]

    crop_height = int(round(width / target_ratio))
    y0 = (height - crop_height) // 2
    return image[y0 : y0 + crop_height, :]


def add_photo(ax, path, target_ratio=1.0):
    image = center_crop_to_ratio(plt.imread(path), target_ratio=target_ratio)
    ax.imshow(image)
    ax.set_axis_off()


def load_masked_depth(raster_path, mask_gdf):
    with rasterio.open(raster_path) as src:
        mask_shapes = mask_gdf.to_crs(src.crs).geometry
        depth, transform = raster_mask(src, mask_shapes, crop=True, filled=True)
        data = depth[0].astype("float64")
        if src.nodata is not None:
            data[data == src.nodata] = np.nan
        data[data < -1e20] = np.nan
        bounds = array_bounds(data.shape[0], data.shape[1], transform)
        raster_crs = src.crs
    return data, bounds, raster_crs


def smooth_depth(depth, sigma=DEPTH_SMOOTHING_SIGMA_PIXELS):
    valid = np.isfinite(depth)
    if sigma <= 0 or not np.any(valid):
        return depth

    filled = np.where(valid, depth, 0.0)
    weights = valid.astype("float64")
    smooth_values = gaussian_filter(filled, sigma=sigma, mode="nearest")
    smooth_weights = gaussian_filter(weights, sigma=sigma, mode="nearest")
    smoothed = np.full_like(depth, np.nan, dtype="float64")
    np.divide(
        smooth_values,
        smooth_weights,
        out=smoothed,
        where=smooth_weights > 0.05,
    )
    smoothed[~valid] = np.nan
    return smoothed


def add_bathymetry_overlay(ax, raster_path, mask_gdf):
    depth, depth_bounds, depth_crs = load_masked_depth(raster_path, mask_gdf)
    max_depth = np.ceil(np.nanpercentile(depth, 98) * 2) / 2
    depth = smooth_depth(depth)
    left, bottom, right, top = depth_bounds
    image = ax.imshow(
        depth,
        extent=[left, right, bottom, top],
        origin="upper",
        cmap="YlGnBu",
        vmin=0,
        vmax=max_depth,
        alpha=0.68,
        interpolation="bilinear",
        resample=True,
        zorder=1,
    )
    ax.add_patch(
        mpatches.Rectangle(
            (0.878, 0.065),
            0.098,
            0.30,
            transform=ax.transAxes,
            facecolor="white",
            edgecolor="none",
            alpha=0.50,
            zorder=2,
        )
    )
    cax = ax.inset_axes([0.913, 0.095, 0.017, 0.215])
    cax.set_zorder(20)
    cax.patch.set_alpha(0)
    cbar = plt.colorbar(image, cax=cax, orientation="vertical", extend="max")
    cbar.set_ticks(np.arange(0, max_depth + 0.001, 0.5))
    cbar.ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))
    cbar.ax.tick_params(labelsize=7.2, colors="black", length=2.4, pad=1.8)
    cbar.outline.set_edgecolor("black")
    for tick_label in cbar.ax.get_yticklabels():
        tick_label.set_color("black")
        tick_label.set_fontweight("medium")
    ax.text(
        0.927,
        0.333,
        "Depth (m)",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=8.0,
        color="black",
        fontweight="medium",
        zorder=21,
    )
    return image


study_area_native = gpd.read_file(STUDY_AREA)
study_crs = study_area_native.crs
cadaster_native = gpd.read_file(CADASTER).to_crs(study_crs)
zoom_studysite_native = gpd.read_file(ZOOM_STUDYSITE).to_crs(study_crs)
with rasterio.open(DRONE_RASTER) as src:
    drone_crs = src.crs
zoom_studysite_drone = zoom_studysite_native.to_crs(drone_crs)
study_area_wm = study_area_native.to_crs("EPSG:3857")
study_bounds_native = study_area_native.total_bounds
study_bounds_wm = study_area_wm.total_bounds

madagascar_extent = square_bounds(
    bbox_wgs84_to_web_mercator(44.3, -26.0, 52.8, -11.5),
    pad_fraction=0.0,
)
sainte_marie_extent = square_bounds(
    bbox_wgs84_to_web_mercator(49.60, -17.30, 50.10, -16.60),
    pad_fraction=0.0,
)
study_extent = square_bounds(study_bounds_native, pad_fraction=0.0)
zoom_extent = square_bounds(zoom_studysite_native.total_bounds, pad_fraction=0.8, min_size=150)
zoom_extent_drone = transform_bounds(zoom_extent, study_crs, drone_crs)

sainte_marie_box = square_bounds(sainte_marie_extent, pad_fraction=0.02)
study_area_box = square_bounds(study_bounds_wm, pad_fraction=0.15)

fig = plt.figure(figsize=(9, 9), facecolor="white")
grid = GridSpec(
    2,
    1,
    figure=fig,
    left=0.015,
    right=0.985,
    bottom=0.025,
    top=0.975,
    hspace=0.025,
    height_ratios=[2, 1],
)
top_grid = grid[0].subgridspec(2, 3, wspace=0.025, hspace=0.025)
bottom_grid = grid[1].subgridspec(1, 2, wspace=0.025)

ax_madagascar = fig.add_subplot(top_grid[0, 0])
ax_picture_d = fig.add_subplot(top_grid[1, 0])
ax_study_area = fig.add_subplot(top_grid[:, 1:3])
ax_picture_e = fig.add_subplot(bottom_grid[0, 0])
ax_picture_f = fig.add_subplot(bottom_grid[0, 1])

add_basemap(ax_madagascar, madagascar_extent, zoom=7, source=REFERENCE_TILE_SOURCE)
add_red_box(ax_madagascar, sainte_marie_box, linewidth=2.0)
add_panel_label(ax_madagascar, "A")
add_scale_bar(ax_madagascar, 200_000, "200 km", x_fraction=0.42)
add_panel_border(ax_madagascar, color="#ffffff", linewidth=0.8)

ax_sainte_marie = ax_madagascar.inset_axes([0.56, 0.045, 0.40, 0.40])
add_basemap(ax_sainte_marie, sainte_marie_extent, zoom=11, source=REFERENCE_TILE_SOURCE)
add_red_box(ax_sainte_marie, study_area_box, linewidth=1.5)
ax_sainte_marie.add_patch(
    mpatches.Rectangle(
        (0, 0),
        1,
        1,
        transform=ax_sainte_marie.transAxes,
        fill=False,
        edgecolor="black",
        linewidth=1.3,
        zorder=20,
    )
)

add_basemap(
    ax_study_area,
    study_extent,
    zoom=15,
    source=SATELLITE_TILE_SOURCE,
    crs=study_crs,
)
add_bathymetry_overlay(ax_study_area, DEPTH_RASTER, study_area_native)
cadaster_native.plot(
    ax=ax_study_area,
    facecolor=(0, 0, 0, 0.16),
    edgecolor=(0, 0, 0, 0.60),
    linewidth=0.18,
    alpha=1.0,
    zorder=9,
)
zoom_studysite_native.plot(
    ax=ax_study_area,
    facecolor=(1.0, 0.84, 0.31, 0.14),
    edgecolor="none",
    zorder=10.5,
)
zoom_studysite_native.boundary.plot(ax=ax_study_area, color="black", linewidth=1.8, zorder=11)
zoom_studysite_native.boundary.plot(
    ax=ax_study_area,
    color="#ffd54f",
    linewidth=1.0,
    zorder=12,
)
study_area_native.boundary.plot(ax=ax_study_area, color="#e31a1c", linewidth=1.5, zorder=10)
ax_study_area.set_xlim(study_extent[0], study_extent[2])
ax_study_area.set_ylim(study_extent[1], study_extent[3])
ax_study_area.set_aspect("equal", adjustable="box")
ax_study_area.set_axis_off()

ax_zoom_inset = ax_study_area.inset_axes([0.05, 0.62, 0.26, 0.26])
add_rgb_raster(ax_zoom_inset, DRONE_RASTER, extent=zoom_extent_drone)
zoom_studysite_drone.plot(
    ax=ax_zoom_inset,
    facecolor=(1.0, 0.84, 0.31, 0.10),
    edgecolor="none",
    zorder=8,
)
zoom_studysite_drone.boundary.plot(ax=ax_zoom_inset, color="black", linewidth=1.6, zorder=9)
zoom_studysite_drone.boundary.plot(
    ax=ax_zoom_inset,
    color="#ffd54f",
    linewidth=0.9,
    zorder=10,
)
ax_zoom_inset.set_xlim(zoom_extent_drone[0], zoom_extent_drone[2])
ax_zoom_inset.set_ylim(zoom_extent_drone[1], zoom_extent_drone[3])
ax_zoom_inset.set_aspect("equal", adjustable="box")
ax_zoom_inset.set_axis_off()
add_panel_border(ax_zoom_inset, color="#ffffff", linewidth=1.0)

add_panel_label(ax_study_area, "B")
add_scale_bar(ax_study_area, 500, "500 m", x_fraction=0.55)
add_panel_border(ax_study_area, color="#ffffff", linewidth=0.8)

add_photo(ax_picture_d, PICTURE_FILES["D"], target_ratio=1.0)
add_panel_label(ax_picture_d, "C")
add_panel_border(ax_picture_d, color="#ffffff", linewidth=0.8)

add_photo(ax_picture_e, PICTURE_FILES["E"], target_ratio=1.5)
add_panel_label(ax_picture_e, "D")
add_panel_border(ax_picture_e, color="#ffffff", linewidth=0.8)

add_photo(ax_picture_f, PICTURE_FILES["F"], target_ratio=1.5)
add_panel_label(ax_picture_f, "E")
add_panel_border(ax_picture_f, color="#ffffff", linewidth=0.8)

fig.savefig(OUT_FILE, dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)

In [ ]:
knitr::include_graphics("Figures/study_area.png")